# Jev for Contract & Document Risk Triage

## What are we building?

In this notebook, we will build a small contract/document triage system using Jev.

Given a contract clause, the system will determine:

1. Does this clause need human review?
2. What type of clause is it?
3. How important is the review?
4. Does the clause contain a deadline, condition, or time-sensitive requirement?

The goal is not to replace lawyers or provide legal advice.

Instead, Jev will act as a **structured decision layer** that helps route documents to the appropriate review workflow.

In [2]:
import os
from dotenv import load_dotenv 
load_dotenv()
from langchain_typesafe import (
    TypeSafeClassifier,
    Noul,
    Choice,
    Score
)

import pandas as pd

classifier = TypeSafeClassifier()

C:\Users\sarvesh\AppData\Local\Temp\ipykernel_14076\3200623897.py:13: LangChainBetaWarning: The class `TypeSafeClassifier` is in beta. It is actively being worked on, so the API may change.
  classifier = TypeSafeClassifier()


## Why are we using Jev?

A traditional LLM could return something like:

"Yes, this looks like a potentially important termination clause."

But for an automated workflow, we want predictable structured signals.

For example:

```text
needs_review = 0.91
clause_type = termination
priority = high
has_deadline = 0.88

In [3]:
clause = """
Either party may terminate this agreement by providing
30 days written notice to the other party.
"""
response = classifier.invoke({
    "state": clause,
    "questions": {
        "needs_review": Noul(
            instructions="""
            Does this contract clause contain something
            that should potentially be reviewed by a human
            because it creates an important contractual
            obligation, condition, deadline, or risk?
            """
        )
    }
})

review_probability = response.nouls["needs_review"].noul

print("Review probability:", review_probability)

Review probability: 0.96


#### Classifying Type of Contract

In [9]:
response = classifier.invoke({ 
    "state": clause, 
    "questions": { 
        "clause_type": Choice(
            instructions="Identify the primary type of this contract clause. Choose the category that best describes the clause.",
            criteria={ 
                "payment": "Fees, invoicing, billing timelines, and methods.",
                "termination": "Ending the contract, exit conditions, and consequences.",
                "data_protection": "GDPR compliance, security measures, and personal data rules.",
                "confidentiality": "NDAs, proprietary trade secrets, and non-disclosure.",
                "auto_renewal": "Subscription extensions and rollover policies.",
                "liability": "Indemnification caps, damage limitations, and risk allocation.",
                "intellectual_property": "Copyright ownership, licensing rights, and patents.",
                "other": "Any clause that does not match the previous definitions."
            } 
        ) 
    } 
})

clause_type = response.choices["clause_type"].choice

print("Clause type:", clause_type)

Clause type: termination


In [12]:
response = classifier.invoke({
    "state": clause,
    "questions": {
        "review_priority": Score(
            # 1. Pydantic expects 'criteria' as the key name
            # 2. It must be passed as a plain list of strings
            criteria=[
                "low",
                "medium",
                "high"
            ],
            instructions="""
            Assess how important it would be for a human reviewer to examine this clause.
            
            Low: Routine clause with limited potential impact.
            Medium: Clause creates a meaningful obligation, condition, deadline, or business impact.
            High: Clause could create significant financial, legal, data, liability, or contractual consequences.
            """
        )
    }
})
priority = response.scores["review_priority"].score

print("Review priority:", priority)

Review priority: 1.26


#### Detect Deadline

In [13]:
response = classifier.invoke({
    "state": clause,
    "questions": {
        "has_deadline": Noul(
            instructions="""
            Does this clause contain a deadline, notice period,
            expiration period, time limit, or other time-sensitive
            condition?
            """
        )
    }
})

deadline_probability = response.nouls["has_deadline"].noul

print("Deadline probability:", deadline_probability)

Deadline probability: 0.99


#### Running Multiple questions in one cell

In [16]:
response = classifier.invoke({
    "state": clause,
    "questions": {
        "needs_review": Noul(
            instructions="""
            Does this contract clause contain something that
            should potentially be reviewed by a human because
            it creates an important contractual obligation,
            condition, deadline, or risk?
            """
        ),

        "has_deadline": Noul(
            instructions="""
            Does this clause contain a deadline, notice period,
            expiration period, time limit, or other time-sensitive
            condition?
            """
        ),

        # FIX: Changed keyword to 'criteria' and converted the list to a dictionary map
        "clause_type": Choice(
            criteria={
                "payment": "Fees, pricing, invoicing, billing terms, and payments.",
                "termination": "Ending the contract, cancellation, and exit clauses.",
                "data_protection": "Data privacy, security measures, and compliance rules.",
                "confidentiality": "Non-disclosure, secret info, and privacy protections.",
                "auto_renewal": "Extensions, automatic roll-overs, or subscription renewals.",
                "liability": "Indemnifications, liability caps, and legal risks.",
                "intellectual_property": "Copyrights, trademarks, patents, and ownership.",
                "other": "Any clause type that does not fit the descriptions above."
            },
            instructions="""
            Identify the primary type of this contract clause.
            """
        ),

        # FIX: Changed keyword to 'criteria' as required by your Pydantic schema validation
        "review_priority": Score(
            criteria=[
                "low",
                "medium",
                "high"
            ],
            instructions="""
            Assess the importance of human review.

            Low = routine and limited impact.

            Medium = meaningful obligation, condition,
            deadline, or business impact.

            High = potentially significant financial,
            legal, data, liability, or contractual impact.
            """
        )
    }
})

result = {
    "clause": clause,
    "review_probability": response.nouls["needs_review"].noul,
    "deadline_probability": response.nouls["has_deadline"].noul,
    "clause_type": response.choices["clause_type"].choice,
    "review_priority": response.scores["review_priority"].score
}

result

{'clause': '\nEither party may terminate this agreement by providing\n30 days written notice to the other party.\n',
 'review_probability': 0.95,
 'deadline_probability': 0.99,
 'clause_type': 'termination',
 'review_priority': 1.62}